# Product Image Category Classifier
This notebook trains a transfer learning model using the **MobileNetV2** architecture in PyTorch to classify retail products into 5 categories:
- `shoes`
- `bags`
- `electronics`
- `clothing`
- `groceries`

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt

## 1. Load trained model state dictionary

In [ ]:
model_path = '../app/models/product_classifier.pt'
if os.path.exists(model_path):
    checkpoint = torch.load(model_path)
    classes = checkpoint['classes']
    print(f"Loaded model classes: {classes}")
else:
    print("Model file not found. Please run the training script first.")

## 2. Define Model Architecture

In [ ]:
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, 5)
if os.path.exists(model_path):
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print("Model weights loaded successfully!")

## 3. Verify Model Predictions on Synthetic Input

In [ ]:
# Create a dummy green tensor representing groceries
dummy_grocery = np.ones((224, 224, 3), dtype=np.uint8) * 255
for y in range(224):
    for x in range(224):
        dist = (x - 112)**2 + (y - 112)**2
        if dist < 45**2:
            dummy_grocery[y, x] = [34, 139, 34]

tensor = torch.from_numpy(dummy_grocery).permute(2, 0, 1).float().unsqueeze(0) / 255.0

with torch.no_grad():
    outputs = model(tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probs).item()
    confidence = probs[pred_idx].item()
    print(f"Predicted Class: {classes[pred_idx]} with confidence {confidence * 100:.2f}%")